# Week 13 · Notebook 1 — Agents & Tool Calling
**CSCI 250 — Introduction to Artificial Intelligence · Fall 2026**

Build the **agent loop** (reason → act → observe) with **Claude** (explicit loop) and **Gemini** (automatic function calling). 

> Runs in Colab. **Degrades gracefully without a key** — a no-key fallback agent runs the same loop using a tiny rule-based 'model' so you can see the mechanics either way.

## 0. Install + load keys safely

In [ ]:
!pip -q install anthropic google-generativeai flask

In [ ]:
import os, json
try:
    from google.colab import userdata
    for k in ('ANTHROPIC_API_KEY', 'GEMINI_API_KEY'):
        try: os.environ[k] = userdata.get(k)
        except Exception: pass
except Exception:
    pass  # locally: export the keys in your shell
HAVE_CLAUDE = bool(os.environ.get('ANTHROPIC_API_KEY'))
HAVE_GEMINI = bool(os.environ.get('GEMINI_API_KEY'))
print('Claude key:', HAVE_CLAUDE, '| Gemini key:', HAVE_GEMINI)

## 1. Define some tools
A tool is just a Python function plus a JSON **schema** the model reads. We expose two; in A10 you'll add a third of your own.

In [ ]:
def add(a, b):
    """Add two numbers."""
    return a + b

def get_price(item):
    """Look up a menu price in dollars."""
    return {'coffee': 4, 'tea': 3, 'cocoa': 5}.get(item.lower(), 0)

TOOLS = {'add': add, 'get_price': get_price}

TOOL_SCHEMAS = [
  {'name': 'add', 'description': 'Add two numbers.',
   'input_schema': {'type': 'object',
     'properties': {'a': {'type': 'number'}, 'b': {'type': 'number'}},
     'required': ['a', 'b']}},
  {'name': 'get_price', 'description': 'Look up a menu price in dollars.',
   'input_schema': {'type': 'object',
     'properties': {'item': {'type': 'string'}},
     'required': ['item']}},
]
print('tools ready:', list(TOOLS))

## 2. The Claude agent loop (explicit)
While the model asks for tools, we run them and feed results back. The `max_turns` guard keeps the loop bounded — **always** do this.

In [ ]:
def run_claude_agent(user_msg, max_turns=5, verbose=True):
    import anthropic
    client = anthropic.Anthropic()
    messages = [{'role': 'user', 'content': user_msg}]
    for turn in range(max_turns):
        resp = client.messages.create(
            model='claude-sonnet-4-6', max_tokens=600,
            tools=TOOL_SCHEMAS, messages=messages)
        if resp.stop_reason != 'tool_use':
            return ''.join(b.text for b in resp.content if b.type == 'text')
        messages.append({'role': 'assistant', 'content': resp.content})
        results = []
        for b in resp.content:
            if b.type == 'tool_use':
                out = TOOLS[b.name](**b.input)        # ACT
                if verbose: print(f'  [tool] {b.name}({b.input}) -> {out}')
                results.append({'type': 'tool_result',
                    'tool_use_id': b.id, 'content': str(out)})  # OBSERVE
        messages.append({'role': 'user', 'content': results})
    return 'stopped: hit max turns'

## 3. A no-key fallback agent (same loop, fake model)
So the notebook works with **no API key**, here is the *identical* reason→act→observe loop driven by a trivial rule-based 'model'. It proves the mechanics, not the intelligence.

In [ ]:
import re

def fake_model_decide(user_msg):
    """Tiny stand-in 'model': returns tool calls or a final answer."""
    calls = []
    m = re.search(r'(\d+)\s*\+\s*(\d+)', user_msg)
    if m: calls.append(('add', {'a': int(m.group(1)), 'b': int(m.group(2))}))
    for item in ('coffee', 'tea', 'cocoa'):
        if item in user_msg.lower():
            calls.append(('get_price', {'item': item}))
    return calls

def run_fallback_agent(user_msg, max_turns=5):
    calls = fake_model_decide(user_msg)        # REASON
    observations = []
    for name, args in calls[:max_turns]:
        out = TOOLS[name](**args)               # ACT
        print(f'  [tool] {name}({args}) -> {out}')
        observations.append(f'{name}={out}')   # OBSERVE
    if not observations:
        return "(fallback) I had no tool to use for that."
    return '(fallback) results -> ' + ', '.join(observations)

## 4. Run the agent
Same question either way: it needs `add` AND `get_price`, so the agent must choose to call tools and combine the results.

In [ ]:
Q = 'What is 12 + 30, and how much is a coffee?'
if HAVE_CLAUDE:
    print('CLAUDE AGENT:'); print(run_claude_agent(Q))
else:
    print('FALLBACK AGENT (no key):'); print(run_fallback_agent(Q))

## 5. Gemini: automatic function calling
Gemini reads type hints + docstrings and calls plain Python functions for you — the loop is hidden inside the SDK.

In [ ]:
if HAVE_GEMINI:
    import google.generativeai as genai
    genai.configure(api_key=os.environ['GEMINI_API_KEY'])
    model = genai.GenerativeModel('gemini-2.5-flash',
                                  tools=[add, get_price])
    chat = model.start_chat(enable_automatic_function_calling=True)
    print('GEMINI AGENT:'); print(chat.send_message(Q).text)
else:
    print('No Gemini key — skipping (fallback above shows the loop).')

## 6. Your turn (Assignment A10)
1. Add a **third tool** (e.g. a unit converter or word counter) — function, schema, and an entry in `TOOLS`.
2. Ask a question that forces the agent to use it; capture the transcript.
3. Then serve it via Flask (Notebook 2 / `agent_app.py`).
4. Write the 250-word design note (include MCP in your own words).

In [ ]:
# add your third tool here
